# ACC 380K — Homework 3

## Find the context in annual reports

**Assignment version 1.0.0 · September 29, 2026**

A count tells us how often a phrase appears. To see what the company says about it,
we need the surrounding text. Download three Tesla annual reports, find every appearance
of a phrase, and organize the passages by year and appearance number.

Use what you learned in **Notebooks 1–6**: functions, strings, slices, dictionaries,
and `for` loops. The few new package calls and string-method options are explained here.
No later notebook is required.

### Before you begin

1. Choose **File → Save a copy in Drive** and name your copy `lastname_firstname_homework03.ipynb`.
2. Write each function, then run its collapsed **Grading cell**. The checks use small practice
   inputs and do not need an SEC connection. Read missed checks, revise, and rerun both cells.
3. Complete the real-filing application after your three functions work.
4. Before submitting, restart and run your completed notebook in order. Keep grading cells
   unchanged. Download the `.ipynb` file and submit it as directed in Canvas.

In all three problems, keep the supplied function and parameter names, begin with a nonempty
docstring, and **return the result without printing**. Do not change the argument dictionaries.
Use `for` loops, not `while` loops or comprehensions. Work out the algorithm yourself; the
examples illustrate individual tools and expected results, not completed functions.

**Grading:** If an unchanged supplied grading cell reports **CORRECT — full credit earned**
after a clean restart and run, that earns full credit for that coding problem; no later
hidden test overrides it. The final application and short explanation are reviewed separately.


## A tool for downloading annual reports

A **package** is a collection of code someone else has written. We will use **EdgarTools**
to retrieve reports from the SEC's EDGAR system. Run the installation cell once each time
you start a new Colab session. `%pip` is a notebook command that installs a package.


In [ ]:
%pip install -q edgartools==5.58.0


`import edgar` makes the package available under the name `edgar`. The installation name
is `edgartools`, but the name we import is `edgar`. We call its functions with a dot,
such as `edgar.set_identity(...)`.

The SEC asks automated requests to identify who is making them. Replace the placeholder
below with **your own name and email**, then run the cell. Do not use someone else's
identity or post this completed cell in a public repository.


In [ ]:
import edgar

sec_identity = "Your Name your_email@example.com"  # Replace with your own name and email.
if sec_identity == "Your Name your_email@example.com":
    print("Add your own name and email before downloading real filings.")
else:
    edgar.set_identity(sec_identity)


### Package reference: retrieve one filing as a string

An **accession number** is the SEC's identifier for one filing. For example:

```python
filing = edgar.get_by_accession_number("0001628280-24-002390")
text = filing.markdown()
```

The first call gets a filing that EdgarTools can work with. Its `.markdown()` method
retrieves the report and returns a **string**. Markdown is text with formatting characters
such as `#` for headings and `|` in tables. Leave those characters in the string; you do
not need to clean the Markdown. A supplied preview cell later shows the raw text and its formatting.

The supplied dictionary identifies three original Tesla 10-Ks. Its keys are **fiscal years**
(the years covered), not filing years. For example, the 2023 report was filed in 2024.

References: [installation](https://edgartools.readthedocs.io/en/stable/installation/)
and [filing methods](https://edgartools.readthedocs.io/en/stable/api/filing/).


In [ ]:
accessions_by_year = {2023: '0001628280-24-002390', 2024: '0001628280-25-003063', 2025: '0001628280-26-003952'}


### Preview the Markdown (supplied)

After setting your SEC identity above, run this cell to download the 2023 report and
show its first 3,000 characters twice. `print` shows the raw text, including Markdown
symbols; `display(Markdown(...))` renders the formatting. Neither changes the string.
This cell needs an internet connection.


In [ ]:
from IPython.display import Markdown, display

filing = edgar.get_by_accession_number(accessions_by_year[2023])
preview = filing.markdown()[:3000]
print("Raw Markdown:")
print(preview)
print("Rendered Markdown:")
display(Markdown(preview))


## Problem 1 — Download the reports

Write `download_markdown(accessions_by_year)`. Its argument is a dictionary whose keys are
integer fiscal years and whose values are accession-number strings. Return a **new dictionary**
with the same year keys and the Markdown string for each year's filing as its value.

Use a `for` loop and the two EdgarTools calls in the package reference. Retrieve each filing
once. An empty input dictionary should produce `{}`. Assume each accession identifies a real
filing. You do not need to write error-handling code.

Begin the function with a nonempty docstring. The function must return its result without
printing, and must not change the argument dictionary.

The grading cell uses two tiny practice filings in place of the SEC connection. You still
write the actual package calls. Your function downloads the real reports in the application at the end.


In [ ]:
def download_markdown(accessions_by_year):
    """Return each fiscal year's filing as a Markdown string."""
    return None


In [ ]:
#@title Grading cell — Problem 1 { display-mode: "form" }

import contextlib
import copy
import dis
import inspect
import io
import signal
from types import SimpleNamespace

checks = []

def report(label, passed, hint="", detail=""):
    checks.append(bool(passed))
    print(("✓ " if passed else "✗ ") + label + (" — passed" if passed else " — not yet"))
    if not passed:
        if detail:
            print("   " + detail)
        if hint:
            print("   Hint: " + hint)

def contract(name, parameters):
    function = globals().get(name)
    report(name + " is defined", callable(function), "Run the function's work cell first.")
    if not callable(function):
        return
    report("Parameter names: " + ", ".join(parameters),
           list(inspect.signature(function).parameters) == parameters,
           "Keep the function header from the starter.")
    report("Begins with a nonempty docstring", bool(function.__doc__ and function.__doc__.strip()),
           "Keep or improve the triple-quoted description immediately below def.")
    report("Uses a for loop", any(i.opname == "FOR_ITER" for i in dis.get_instructions(function)),
           "Use the for-loop approach required by the problem.")

def call(name, arguments):
    function = globals().get(name)
    if not callable(function):
        return False, None
    output = io.StringIO()
    old_handler = signal.getsignal(signal.SIGALRM)
    def stop(signum, frame):
        raise TimeoutError("The check took too long.")
    try:
        signal.signal(signal.SIGALRM, stop)
        signal.setitimer(signal.ITIMER_REAL, 1)
        with contextlib.redirect_stdout(output):
            result = function(*arguments)
    except Exception as error:
        report("Runs on the practice input", False,
               "Check your variable names and how you use the input, then rerun the work cell.",
               type(error).__name__ + ": " + str(error)[:150])
        return False, None
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_handler)
    if output.getvalue() != "":
        report("Returns without printing", False,
               "Remove print calls from the function; return the dictionary instead.")
    return True, result

def case(label, name, arguments, expected, hint):
    before = copy.deepcopy(arguments)
    ok, result = call(name, arguments)
    if ok:
        detail = "Expected " + repr(expected)[:160] + "; received " + repr(result)[:160]
        if result is None:
            hint = "Replace the starter return None with your result. Check where your return belongs."
        if arguments != before:
            hint = "Build a new result dictionary without changing the input."
            detail += "; the input was changed"
        report(label, exact(result, expected) and arguments == before, hint, detail)
    return ok, result

def exact(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return (len(actual) == len(expected)
                and all(type(k) is type(next(iter(expected))) for k in actual)
                and all(k in actual and exact(actual[k], v) for k, v in expected.items()))
    return actual == expected

def fresh_result(name, arguments):
    ok1, first = call(name, copy.deepcopy(arguments))
    ok2, second = call(name, copy.deepcopy(arguments))
    report("Builds a fresh result on each call", ok1 and ok2 and type(first) is dict
           and type(second) is dict and first is not second,
           "Create the result inside the function, not in a variable outside it.")

def finish(number):
    passed = bool(checks) and all(checks)
    globals()["hw3_p" + str(number) + "_ready"] = passed
    print()
    print("Problem " + str(number) + " correct:", passed)
    print("Problem " + str(number) + (": CORRECT — full credit earned." if passed else ": NOT YET CORRECT"))
    if not passed:
        print("Read the missed checks and hints. Edit and rerun your work cell, then this cell.")


contract("download_markdown", ["accessions_by_year"])

# Practice filings test the direct package calls without contacting the SEC.
class PracticeFiling:
    def __init__(self, text):
        self.text = text
        self.downloads = 0
    def markdown(self):
        self.downloads += 1
        return self.text

practice_filings = {"A": PracticeFiling("# Annual report A"), "B": PracticeFiling("# Annual report B")}
lookups = []
def practice_lookup(accession):
    lookups.append(accession)
    return practice_filings[accession]

function = globals().get("download_markdown")
if callable(function):
    scope = function.__globals__
    saved_edgar = scope.get("edgar")
    scope["edgar"] = SimpleNamespace(get_by_accession_number=practice_lookup)
    try:
        case("Stores each year's Markdown under that year", "download_markdown",
             [{2023: "A", 2024: "B"}], {2023: "# Annual report A", 2024: "# Annual report B"},
             "Check what your loop uses as each key and each value.")
        report("Looks up each supplied accession once", sorted(lookups) == ["A", "B"],
               "Use edgar.get_by_accession_number for each accession in the argument.")
        report("Calls .markdown() once on each filing", all(f.downloads == 1 for f in practice_filings.values()),
               "Check the method name and parentheses in the package reference.")
        case("No filings gives an empty dictionary", "download_markdown", [{}], {},
             "What result should remain when the loop runs zero times?")
        fresh_result("download_markdown", [{2023: "A"}])
    finally:
        if saved_edgar is None:
            scope.pop("edgar", None)
        else:
            scope["edgar"] = saved_edgar
finish(1)


## String-method reference

`text.count(phrase)` returns the number of appearances of `phrase` in `text`.
`text.find(phrase, start)` searches **at or after** position `start`. Its result is a position
measured from the beginning of the **whole string**, not from `start`. As before, `-1` means
no match was found. Predict the outputs, then run this example:


In [ ]:
sample = "red blue red"
print(sample.count("red"))
print(sample.find("red", 5))


The outputs are `2` and `9`. The search starts at position 5 but reports the match's
position in the whole string.

In this homework, matches are **exact, case-sensitive substrings**: `"cat"` matches inside
`"cats"`, but not `"Cat"`. Do not change capitalization before searching.


## Problem 2 — Collect every appearance and its context

Write `collect_contexts(text, search_phrase, context_length)`. Assume `text` is a string,
`search_phrase` is a **nonempty** string, and `context_length` is an integer of 0 or more.
You do not need to check these input types.

Return a **new dictionary** with appearance numbers `1`, `2`, `3`, and so on as its keys,
in the order found. Each value must contain the **whole matched phrase**, plus up to
`context_length` characters before it and after it, taken from the original string.

Required tools: call `.count()` once before a `for` loop that runs once per counted
appearance. Inside the loop, use `.find()` with a starting position to locate that
appearance, then use a slice to extract its context. Work out how to update the starting
position so you neither repeat nor skip matches. Do not use a `while` loop.

Your result must handle these cases:

- A match near either end uses only the characters available; it must not wrap around.
- With a context length of 0, each value is just the matched phrase.
- If there are no matches, including when `text` is empty, return `{}`.
- Keep a passage for every appearance, even when two passages contain the same text.

Count spaces, punctuation, line breaks, and Markdown symbols as characters too.
For example, `collect_contexts("xx cat yy cat zz", "cat", 2)` should return
`{1: "x cat y", 2: "y cat z"}`.

Begin the function with a nonempty docstring. The function must return its result without printing.


In [ ]:
def collect_contexts(text, search_phrase, context_length):
    """Return numbered context passages for every appearance of the phrase."""
    return None


In [ ]:
#@title Grading cell — Problem 2 { display-mode: "form" }

import contextlib
import copy
import dis
import inspect
import io
import signal
from types import SimpleNamespace

checks = []

def report(label, passed, hint="", detail=""):
    checks.append(bool(passed))
    print(("✓ " if passed else "✗ ") + label + (" — passed" if passed else " — not yet"))
    if not passed:
        if detail:
            print("   " + detail)
        if hint:
            print("   Hint: " + hint)

def contract(name, parameters):
    function = globals().get(name)
    report(name + " is defined", callable(function), "Run the function's work cell first.")
    if not callable(function):
        return
    report("Parameter names: " + ", ".join(parameters),
           list(inspect.signature(function).parameters) == parameters,
           "Keep the function header from the starter.")
    report("Begins with a nonempty docstring", bool(function.__doc__ and function.__doc__.strip()),
           "Keep or improve the triple-quoted description immediately below def.")
    report("Uses a for loop", any(i.opname == "FOR_ITER" for i in dis.get_instructions(function)),
           "Use the for-loop approach required by the problem.")

def call(name, arguments):
    function = globals().get(name)
    if not callable(function):
        return False, None
    output = io.StringIO()
    old_handler = signal.getsignal(signal.SIGALRM)
    def stop(signum, frame):
        raise TimeoutError("The check took too long.")
    try:
        signal.signal(signal.SIGALRM, stop)
        signal.setitimer(signal.ITIMER_REAL, 1)
        with contextlib.redirect_stdout(output):
            result = function(*arguments)
    except Exception as error:
        report("Runs on the practice input", False,
               "Check your variable names and how you use the input, then rerun the work cell.",
               type(error).__name__ + ": " + str(error)[:150])
        return False, None
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_handler)
    if output.getvalue() != "":
        report("Returns without printing", False,
               "Remove print calls from the function; return the dictionary instead.")
    return True, result

def case(label, name, arguments, expected, hint):
    before = copy.deepcopy(arguments)
    ok, result = call(name, arguments)
    if ok:
        detail = "Expected " + repr(expected)[:160] + "; received " + repr(result)[:160]
        if result is None:
            hint = "Replace the starter return None with your result. Check where your return belongs."
        if arguments != before:
            hint = "Build a new result dictionary without changing the input."
            detail += "; the input was changed"
        report(label, exact(result, expected) and arguments == before, hint, detail)
    return ok, result

def exact(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return (len(actual) == len(expected)
                and all(type(k) is type(next(iter(expected))) for k in actual)
                and all(k in actual and exact(actual[k], v) for k, v in expected.items()))
    return actual == expected

def fresh_result(name, arguments):
    ok1, first = call(name, copy.deepcopy(arguments))
    ok2, second = call(name, copy.deepcopy(arguments))
    report("Builds a fresh result on each call", ok1 and ok2 and type(first) is dict
           and type(second) is dict and first is not second,
           "Create the result inside the function, not in a variable outside it.")

def finish(number):
    passed = bool(checks) and all(checks)
    globals()["hw3_p" + str(number) + "_ready"] = passed
    print()
    print("Problem " + str(number) + " correct:", passed)
    print("Problem " + str(number) + (": CORRECT — full credit earned." if passed else ": NOT YET CORRECT"))
    if not passed:
        print("Read the missed checks and hints. Edit and rerun your work cell, then this cell.")


contract("collect_contexts", ["text", "search_phrase", "context_length"])
cases = [
    ("Two different passages", ["xx cat yy cat zz", "cat", 2], {1: "x cat y", 2: "y cat z"},
     "Does your next search move past the match you just found?"),
    ("Three nearby appearances", ["cat cat cat", "cat", 2],
     {1: "cat c", 2: "t cat c", 3: "t cat"},
     "Do not skip another appearance just because it falls inside a previous passage."),
    ("Match at the beginning", ["cat naps", "cat", 2], {1: "cat n"},
     "A negative slice start counts from the end. Check the start before slicing."),
    ("Match at the end", ["a cat", "cat", 2], {1: "a cat"},
     "Include the entire phrase as well as the requested context."),
    ("Zero context", ["cat cat", "cat", 0], {1: "cat", 2: "cat"},
     "Zero context still includes the whole matched phrase."),
    ("Long context can include nearby appearances", ["cat cat", "cat", 30], {1: "cat cat", 2: "cat cat"},
     "The next search is about the next match, not the end of the surrounding passage."),
    ("Case-sensitive matching", ["Cat cat CAT", "cat", 1], {1: " cat "},
     "Search the original string with the exact phrase, without changing capitalization."),
    ("No matches", ["solar roofs", "cat", 3], {}, "A count of 0 means no appearances to collect."),
    ("Empty text", ["", "cat", 3], {}, "An empty string has no matches."),
]
for label, arguments, expected, hint in cases:
    ok, result = case(label, "collect_contexts", arguments, expected, hint)
    if ok and isinstance(result, dict):
        if not (all(type(k) is int for k in result) and list(result) == list(range(1, len(result) + 1))):
            report("Appearance keys are integers starting at 1", False,
                   "Use appearance numbers, not character positions or string keys.")

class TrackedText(str):
    def count(self, phrase, *args):
        calls.append(("count", phrase, args))
        return super().count(phrase, *args)
    def find(self, phrase, *args):
        calls.append(("find", phrase, args))
        return super().find(phrase, *args)

calls = []
ok, result = call("collect_contexts", [TrackedText("cat xx cat"), "cat", 1])
if ok:
    report("Counts first, then finds each match with a starting position",
           [entry[0] for entry in calls] == ["count", "find", "find"]
           and all(entry[1] == "cat" for entry in calls)
           and all(len(entry[2]) == 1 for entry in calls if entry[0] == "find"),
           "Use .count() before your loop and .find() with a starting position inside it.")
fresh_result("collect_contexts", ["cat", "cat", 1])
finish(2)


## A dictionary can contain other dictionaries

A dictionary's values can themselves be dictionaries. For example:

```python
room_scores = {101: {"morning": 4, "afternoon": 7}, 202: {}}
```

`room_scores[101]` is the inner dictionary for room 101. `room_scores[101]["afternoon"]`
is `7`. `len(room_scores[101])` is `2`, the number of entries in that inner dictionary.
Room 202 has an entry in the outer dictionary but no entries in its inner dictionary.


## Problem 3 — Organize the passages by year

Write `contexts_by_year(texts_by_year, search_phrase, context_length)`. The first argument
is a dictionary of integer years and strings, like the result of Problem 1. The other
arguments have the same meaning and assumptions as in Problem 2.

Return a **new nested dictionary**. Its outer keys are the years. Each year's value is a
separate dictionary of numbered context passages. Use a `for` loop and call your
`collect_contexts` function once for each year's text.

Keep every year, including years without matches. Appearance numbering restarts at 1 in
each year. An empty input dictionary produces `{}`. For example:

```python
contexts_by_year({2023: "cat dog cat", 2024: "dog"}, "cat", 1)
```

should return `{2023: {1: "cat ", 2: " cat"}, 2024: {}}`.

Begin the function with a nonempty docstring. The function must return its result without
printing, and must not change the argument dictionary.


In [ ]:
def contexts_by_year(texts_by_year, search_phrase, context_length):
    """Return the numbered context passages for each year."""
    return None


In [ ]:
#@title Grading cell — Problem 3 { display-mode: "form" }

import contextlib
import copy
import dis
import inspect
import io
import signal
from types import SimpleNamespace

checks = []

def report(label, passed, hint="", detail=""):
    checks.append(bool(passed))
    print(("✓ " if passed else "✗ ") + label + (" — passed" if passed else " — not yet"))
    if not passed:
        if detail:
            print("   " + detail)
        if hint:
            print("   Hint: " + hint)

def contract(name, parameters):
    function = globals().get(name)
    report(name + " is defined", callable(function), "Run the function's work cell first.")
    if not callable(function):
        return
    report("Parameter names: " + ", ".join(parameters),
           list(inspect.signature(function).parameters) == parameters,
           "Keep the function header from the starter.")
    report("Begins with a nonempty docstring", bool(function.__doc__ and function.__doc__.strip()),
           "Keep or improve the triple-quoted description immediately below def.")
    report("Uses a for loop", any(i.opname == "FOR_ITER" for i in dis.get_instructions(function)),
           "Use the for-loop approach required by the problem.")

def call(name, arguments):
    function = globals().get(name)
    if not callable(function):
        return False, None
    output = io.StringIO()
    old_handler = signal.getsignal(signal.SIGALRM)
    def stop(signum, frame):
        raise TimeoutError("The check took too long.")
    try:
        signal.signal(signal.SIGALRM, stop)
        signal.setitimer(signal.ITIMER_REAL, 1)
        with contextlib.redirect_stdout(output):
            result = function(*arguments)
    except Exception as error:
        report("Runs on the practice input", False,
               "Check your variable names and how you use the input, then rerun the work cell.",
               type(error).__name__ + ": " + str(error)[:150])
        return False, None
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_handler)
    if output.getvalue() != "":
        report("Returns without printing", False,
               "Remove print calls from the function; return the dictionary instead.")
    return True, result

def case(label, name, arguments, expected, hint):
    before = copy.deepcopy(arguments)
    ok, result = call(name, arguments)
    if ok:
        detail = "Expected " + repr(expected)[:160] + "; received " + repr(result)[:160]
        if result is None:
            hint = "Replace the starter return None with your result. Check where your return belongs."
        if arguments != before:
            hint = "Build a new result dictionary without changing the input."
            detail += "; the input was changed"
        report(label, exact(result, expected) and arguments == before, hint, detail)
    return ok, result

def exact(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return (len(actual) == len(expected)
                and all(type(k) is type(next(iter(expected))) for k in actual)
                and all(k in actual and exact(actual[k], v) for k, v in expected.items()))
    return actual == expected

def fresh_result(name, arguments):
    ok1, first = call(name, copy.deepcopy(arguments))
    ok2, second = call(name, copy.deepcopy(arguments))
    report("Builds a fresh result on each call", ok1 and ok2 and type(first) is dict
           and type(second) is dict and first is not second,
           "Create the result inside the function, not in a variable outside it.")

def finish(number):
    passed = bool(checks) and all(checks)
    globals()["hw3_p" + str(number) + "_ready"] = passed
    print()
    print("Problem " + str(number) + " correct:", passed)
    print("Problem " + str(number) + (": CORRECT — full credit earned." if passed else ": NOT YET CORRECT"))
    if not passed:
        print("Read the missed checks and hints. Edit and rerun your work cell, then this cell.")


contract("contexts_by_year", ["texts_by_year", "search_phrase", "context_length"])
case("All years, including a year without matches", "contexts_by_year",
     [{2023: "cat dog cat", 2024: "dog", 2025: "a cat"}, "cat", 1],
     {2023: {1: "cat ", 2: " cat"}, 2024: {}, 2025: {1: " cat"}},
     "Check which value you store under each year. A year without matches must still appear.")
case("No years", "contexts_by_year", [{}, "cat", 1], {}, "An empty input gives an empty result.")
ok, result = case("Numbering restarts in each year", "contexts_by_year",
                  [{2021: "cat", 2022: "cat"}, "cat", 0],
                  {2021: {1: "cat"}, 2022: {1: "cat"}}, "Each year's appearances start at 1.")
if ok and isinstance(result, dict) and 2021 in result and 2022 in result:
    report("Each year has its own dictionary", result[2021] is not result[2022],
           "Do not reuse one shared inner dictionary for multiple years.")
function = globals().get("contexts_by_year")
if callable(function):
    scope = function.__globals__
    saved_collect = scope.get("collect_contexts")
    helper_calls = []
    def practice_collect(text, phrase, distance):
        helper_calls.append((text, phrase, distance))
        return {1: text}
    scope["collect_contexts"] = practice_collect
    try:
        ok, result = call("contexts_by_year", [{2023: "A", 2024: "B"}, "X", 7])
        report("Uses collect_contexts once per year",
               ok and sorted(helper_calls) == [("A", "X", 7), ("B", "X", 7)]
               and result == {2023: {1: "A"}, 2024: {1: "B"}},
               "Reuse your Problem 2 function with each year's text and the requested phrase and context length.")
    finally:
        if saved_collect is None:
            scope.pop("collect_contexts", None)
        else:
            scope["collect_contexts"] = saved_collect
fresh_result("contexts_by_year", [{2023: "cat"}, "cat", 1])
finish(3)


## Apply your functions to Tesla's reports

After all three grading cells say `CORRECT`, run the cell below. It calls **your**
`download_markdown` function, then passes its result directly to **your** `contexts_by_year`
function. The result contains passages around `"Musk"`, with up to 50 characters on each side,
organized by year. Save it as `filing_contexts` and display it.

This step needs internet access and your SEC identity from the setup cell. If access is
denied (403), check your identity and ask for help if it persists. If the SEC asks you to
slow down (429), wait a few minutes before retrying. Do not repeatedly rerun downloads.
The three function checks work even when the SEC connection is unavailable.


In [ ]:
filing_contexts = contexts_by_year(download_markdown(accessions_by_year), "Musk", 50)
filing_contexts


### Search and read the results

Use `filing_contexts` to print each year and its number of appearances. Printing here
is fine: this is outside your functions. The outer keys should be `2023`, `2024`, and `2025`.

Choose a year with at least one match and print its first passage by using both dictionary
keys. In the text cell, explain in one or two sentences what that passage says about Musk.
If the passage is too short to understand, increase the context length in the call above
and rerun it. Do not infer financial consequences beyond what the passage actually says.


In [ ]:
# Report each year's count and print one passage from filing_contexts.


Write your explanation here.


## Before submitting

- All three unchanged grading cells report `CORRECT` after you rerun your function cells.
- Your application shows a count for each year and one passage, with a short explanation.
- You can explain how you advance to the next match, choose slice boundaries, and keep
  each year's appearances separate.

If the SEC connection remains unavailable, tell the instructor or TA rather than inventing
filing text or counts. The function checks still demonstrate your coding work.
